# Lesson 2A: Hierarchical Clustering - Theory

## Introduction

You work for a biology research lab studying evolutionary relationships between species. You have DNA sequences from 100 organisms, but you do not know how many distinct groups exist or what the evolutionary tree looks like.

K-Means requires specifying k in advance, but you do not know the right number of clusters! Moreover, you want to see the **hierarchical structure** - which species are closely related, which share common ancestors, and how the entire evolutionary tree is organized.

This is exactly what **Hierarchical Clustering** solves. It builds a tree (dendrogram) showing relationships at all levels, letting you choose any number of clusters after the fact.

**Real-world applications**:
- **Evolutionary biology**: Build phylogenetic trees from DNA sequences
- **Document clustering**: Organize documents into topic hierarchies
- **Market segmentation**: Create customer hierarchies (segment → sub-segment)
- **Gene expression analysis**: Group genes by expression patterns
- **Social network analysis**: Find community structures

In this lesson, we will:
1. Understand agglomerative vs divisive hierarchical clustering
2. Learn different linkage methods (single, complete, average, Ward)
3. Derive the mathematics of linkage criteria
4. Build dendrograms and interpret them
5. Implement hierarchical clustering from scratch
6. Apply to gene expression data

Then in Lesson 2B, we will use scikit-learn for production and apply to real biological datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.datasets import make_blobs
from typing import List, Tuple
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## Hierarchical Clustering Overview

### Two Approaches

**1. Agglomerative (Bottom-Up)**:
- Start: Each point is its own cluster
- Repeat: Merge two closest clusters
- Stop: All points in one cluster
- Most common approach

**2. Divisive (Top-Down)**:
- Start: All points in one cluster
- Repeat: Split cluster into two parts
- Stop: Each point is its own cluster
- Less common (computationally expensive)

We will focus on **agglomerative** hierarchical clustering.

### Algorithm Steps (Agglomerative)

**Input**: Dataset $X$ with $n$ points

**Output**: Dendrogram showing hierarchical structure

1. **Initialize**: Each point $x_i$ is a cluster $C_i = \{x_i\}$
2. **Repeat** until one cluster remains:
   - Find two closest clusters $C_i$ and $C_j$
   - Merge them: $C_{new} = C_i \cup C_j$
   - Remove $C_i$ and $C_j$, add $C_{new}$
   - Record merge in dendrogram
3. **Output**: Tree structure (dendrogram)

### Key Question: How to Measure Distance Between Clusters?

This is answered by **linkage methods**!

## Linkage Methods

Given two clusters $C_i$ and $C_j$, how do we measure distance $d(C_i, C_j)$?

### 1. Single Linkage (MIN)

Distance = minimum distance between any two points:

$$d(C_i, C_j) = \min_{x \in C_i, y \in C_j} d(x, y)$$

**Properties**:
- Tends to create long, chain-like clusters
- Sensitive to noise and outliers
- Good for: Finding arbitrary-shaped clusters

### 2. Complete Linkage (MAX)

Distance = maximum distance between any two points:

$$d(C_i, C_j) = \max_{x \in C_i, y \in C_j} d(x, y)$$

**Properties**:
- Tends to create compact, spherical clusters
- Less sensitive to outliers than single linkage
- Good for: Well-separated spherical clusters

### 3. Average Linkage (UPGMA)

Distance = average distance between all pairs:

$$d(C_i, C_j) = \frac{1}{|C_i| |C_j|} \sum_{x \in C_i} \sum_{y \in C_j} d(x, y)$$

**Properties**:
- Balanced approach between single and complete
- Good general-purpose method
- Good for: Most applications

### 4. Ward's Method

Minimize increase in total within-cluster variance:

$$d(C_i, C_j) = \frac{|C_i| |C_j|}{|C_i| + |C_j|} \|\mu_i - \mu_j\|^2$$

where $\mu_i$ and $\mu_j$ are cluster centroids.

**Properties**:
- Minimizes variance (similar to K-Means objective)
- Tends to create equal-sized clusters
- Good for: Most applications, especially with Euclidean distance

In [ ]:
# Implement simple hierarchical clustering from scratch
class SimpleHierarchicalClustering:
    """
    Simple agglomerative hierarchical clustering implementation.
    """

    def __init__(self, linkage='average'):
        """
        Initialize hierarchical clustering.

        Args:
            linkage: Linkage criterion ('single', 'complete', 'average')
        """
        self.linkage = linkage
        self.merge_history = []

    def fit(self, X: NDArray):
        """
        Perform agglomerative hierarchical clustering.

        Args:
            X: Data of shape (n_samples, n_features)
        """
        n_samples = X.shape[0]

        # Initialize: each point is its own cluster
        clusters = [[i] for i in range(n_samples)]

        # Merge until one cluster remains
        while len(clusters) > 1:
            # Find two closest clusters
            min_dist = np.inf
            merge_i, merge_j = 0, 1

            for i in range(len(clusters)):
                for j in range(i + 1, len(clusters)):
                    dist = self._cluster_distance(X, clusters[i], clusters[j])
                    if dist < min_dist:
                        min_dist = dist
                        merge_i, merge_j = i, j

            # Merge clusters
            new_cluster = clusters[merge_i] + clusters[merge_j]

            # Record merge
            self.merge_history.append({
                'clusters': (clusters[merge_i].copy(), clusters[merge_j].copy()),
                'distance': min_dist,
                'size': len(new_cluster)
            })

            # Update cluster list
            clusters = [c for idx, c in enumerate(clusters) if idx not in [merge_i, merge_j]]
            clusters.append(new_cluster)

        return self

    def _cluster_distance(self, X: NDArray, cluster_i: List[int], cluster_j: List[int]) -> float:
        """
        Compute distance between two clusters based on linkage criterion.

        Args:
            X: Data matrix
            cluster_i: Indices of points in first cluster
            cluster_j: Indices of points in second cluster

        Returns:
            Distance between clusters
        """
        distances = []

        for i in cluster_i:
            for j in cluster_j:
                dist = np.linalg.norm(X[i] - X[j])
                distances.append(dist)

        distances = np.array(distances)

        if self.linkage == 'single':
            return np.min(distances)
        elif self.linkage == 'complete':
            return np.max(distances)
        elif self.linkage == 'average':
            return np.mean(distances)
        else:
            raise ValueError(f"Unknown linkage: {self.linkage}")

print("✅ Hierarchical clustering class implemented!")

In [ ]:
# Test hierarchical clustering
print("Creating synthetic dataset with hierarchical structure...")

X, y_true = make_blobs(n_samples=50, centers=3, n_features=2,
                       cluster_std=0.5, random_state=42)

print(f"Dataset: {X.shape[0]} samples\n")

# Try different linkage methods
for linkage_type in ['single', 'complete', 'average']:
    print(f"Testing {linkage_type} linkage...")
    hc = SimpleHierarchicalClustering(linkage=linkage_type)
    hc.fit(X)
    print(f"  Completed {len(hc.merge_history)} merges")
    print(f"  Final merge distance: {hc.merge_history[-1]['distance']:.4f}\n")

print("✅ Hierarchical clustering working on all linkage methods!")